In [2]:
%pip install shap

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/751.9 kB ? eta -:--:--
   --------------------------- ------------ 524.3/751.9 kB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 751.9/751.9 kB 2.2 MB/s  0:00:00

   ---------------------------------------- 0/2 [slicer]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- -------

In [1]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 0: environment check
import importlib, sys
for pkg in ["numpy", "pandas", "lightgbm", "shap", "sklearn", "geopandas", "matplotlib", "scipy", "joblib"]:
    m = importlib.import_module(pkg)
    print(f"{pkg:12s} {getattr(m, '__version__', '?')}")
print("Python", sys.version.split()[0])

numpy        2.3.5
pandas       2.3.3
lightgbm     4.7.0
shap         0.54.1
sklearn      1.7.2
geopandas    1.1.4
matplotlib   3.10.6
scipy        1.16.3
joblib       1.5.2
Python 3.13.9


In [2]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 1: settings + inputs + checks
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib
import lightgbm as lgb
from scipy.stats import spearmanr

# ---------------- SETTINGS ----------------
ROOT = Path(r"D:\Siam\Thesis_GeoAI")
MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
GRID = ROOT / "02_boundary_grid" / "grid_250m_v01.gpkg"
LABELS = ROOT / "03_features" / "Label_Flooded_Event_v02.csv"
V03 = ROOT / "05_models" / "final_v03"
V02_PRED = ROOT / "05_models" / "final_v02" / "hazard_predictions_v02.csv"
OUT = ROOT / "05_models" / "phase4_v01"
FIG = OUT / "figures"

FEATURES = ["Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m", "Cropland_Fraction"]
TARGET = "Flooded_Event"
BLOCK_SIZE_M = 15000
N_ENSEMBLE = 50
SEED = 42
N_BINS = 40                  # bins for SHAP dependence curves
CORR_FLAG = 0.7              # |Spearman| flag for correlated features
LGBM_PATH_FILTER = None      # set a substring only if Cell 1 finds extra LGBM objects (e.g. fold models)
EXPECTED_MODEL_CELLS = 160874
EXPECTED_TRAIN_ROWS = 148719
EXPECTED_TRAIN_CELLS = 49573
EXPECTED_BLOCKS = 73
# ------------------------------------------
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

def header(p):
    return list(pd.read_csv(p, nrows=0).columns)

# 1) artefacts in final_v03
print("== final_v03 files ==")
for p in sorted(V03.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(V03)):55s} {p.stat().st_size / 1e6:8.1f} MB")
cfg = json.loads((V03 / "model_config_v03.json").read_text())
print("\n== model_config_v03.json (first 2500 chars) ==")
print(json.dumps(cfg, indent=1, default=str)[:2500])

# 2) master + feature columns (auto-locate any column missing from master)
mcols = header(MASTER)
need = ["Grid_ID", "Is_Water"] + FEATURES
optional = [c for c in ["Population_Count", "District"] if c in mcols]
missing = [c for c in need if c not in mcols]
extra_src = {}
for p in sorted((ROOT / "03_features").glob("*.csv")):
    if not missing:
        break
    h = header(p)
    for c in missing:
        if c in h and "Grid_ID" in h and c not in extra_src:
            extra_src[c] = p
still = [c for c in missing if c not in extra_src]
assert not still, f"Columns not found in master or 03_features: {still} -> paste this to Claude"
master = pd.read_csv(MASTER, usecols=[c for c in need + optional if c in mcols])
for c, p in extra_src.items():
    add = pd.read_csv(p, usecols=["Grid_ID", c])
    assert add["Grid_ID"].is_unique, f"duplicate Grid_ID in {p.name}"
    master = master.merge(add, on="Grid_ID", how="left", validate="1:1")
    print(f"{c} taken from {p.name}")
assert master["Grid_ID"].is_unique
print("\nmaster rows:", len(master), "| optional cols:", optional)

cells = master.loc[master["Is_Water"] == 0].copy()
assert len(cells) == EXPECTED_MODEL_CELLS, f"model cells {len(cells)} != {EXPECTED_MODEL_CELLS}"
print("NaN per feature (model cells):", cells[FEATURES].isna().sum().to_dict())
print(cells[FEATURES].describe(percentiles=[0.01, 0.5, 0.99]).T[["min", "1%", "50%", "99%", "max"]].round(3))
assert (cells["Slope_Mean_deg"].dropna() >= 0).all()
assert (cells["Distance_to_River_m"].dropna() >= 0).all()
assert cells["Cropland_Fraction"].dropna().between(0, 1).all()
assert cells["Elevation_Mean_m"].dropna().between(-20, 150).all(), "implausible elevation range"

# 3) final v03 predictions
hz = pd.read_csv(V03 / "hazard_predictions_v03.csv")
need_hz = ["Grid_ID", "Hazard_Prob_LGBM", "Is_Observable", "AOA_DI", "Is_Within_AOA"]
assert all(c in hz.columns for c in need_hz), hz.columns.tolist()
assert hz["Grid_ID"].is_unique and len(hz) == EXPECTED_MODEL_CELLS
cells = cells.merge(hz[need_hz], on="Grid_ID", how="left", validate="1:1")
assert cells["Hazard_Prob_LGBM"].notna().all()

# 4) grid centroids (for blocks only; maps use polygons)
grid = gpd.read_file(GRID)
assert "Grid_ID" in grid.columns and grid["Grid_ID"].is_unique
assert grid.crs is not None and grid.crs.to_epsg() == 32646, grid.crs
cent = grid.geometry.centroid
grid_xy = pd.DataFrame({"Grid_ID": grid["Grid_ID"].values, "X_m": cent.x.values, "Y_m": cent.y.values})
print("grid polygons:", len(grid))
cells = cells.merge(grid_xy, on="Grid_ID", how="left", validate="1:1")
assert cells["X_m"].notna().all()
cells = cells.reset_index(drop=True)

# 5) locate fitted LGBMClassifier(s) inside the joblib bundle
bundle = joblib.load(V03 / "final_models_v03.joblib")
print("\njoblib type:", type(bundle).__name__, "| keys:", list(bundle.keys()) if isinstance(bundle, dict) else "-")

def _fitted(o):
    try:
        o.booster_
        return True
    except Exception:
        return False

def find_lgbm(obj, path="bundle", out=None, seen=None, depth=0):
    out = [] if out is None else out
    seen = set() if seen is None else seen
    if id(obj) in seen or depth > 6:
        return out
    seen.add(id(obj))
    if isinstance(obj, lgb.LGBMClassifier):
        if _fitted(obj):
            out.append((path, obj))
        return out
    if isinstance(obj, dict):
        for k, v in obj.items():
            find_lgbm(v, f"{path}[{k!r}]", out, seen, depth + 1)
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            find_lgbm(v, f"{path}[{i}]", out, seen, depth + 1)
    else:
        for a in ("calibrated_classifiers_", "estimator", "base_estimator", "estimator_", "steps"):
            try:
                v = getattr(obj, a, None)
            except Exception:
                v = None
            if v is not None and not isinstance(v, (str, int, float, bool)):
                find_lgbm(v, f"{path}.{a}", out, seen, depth + 1)
    return out

found = find_lgbm(bundle)
print("fitted LGBMClassifier objects found:")
for p, m in found:
    print(f"  {p} | n_features={m.n_features_in_} | names={m.booster_.feature_name()} | trees={m.booster_.num_trees()}")
cand = [(p, m) for p, m in found if m.n_features_in_ == len(FEATURES)]
named = [(p, m) for p, m in cand if set(m.booster_.feature_name()) == set(FEATURES)]
members = named if named else cand
if LGBM_PATH_FILTER:
    members = [(p, m) for p, m in members if LGBM_PATH_FILTER.lower() in p.lower()]
assert len(members) >= 1, "No 4-feature fitted LGBM found -> paste the list above to Claude"
if named:
    FEATURES = list(members[0][1].booster_.feature_name())   # use model's column order
    assert all(list(m.booster_.feature_name()) == FEATURES for _, m in members)
else:
    print("WARNING: model has generic feature names; FEATURES order assumed (checked below by Spearman)")
print("USING", len(members), "LGBM member(s); feature order:", FEATURES)

def X_for(model, df):
    if list(model.booster_.feature_name()) == FEATURES:
        return df[FEATURES]
    return df[FEATURES].to_numpy()

raw_final = np.mean([m.predict(X_for(m, cells), raw_score=True) for _, m in members], axis=0)
rho_rep = spearmanr(raw_final, cells["Hazard_Prob_LGBM"]).correlation
print(f"Reproduction check Spearman(raw LGBM margin, Hazard_Prob_LGBM) = {rho_rep:.4f}")
assert rho_rep > 0.95, "Loaded model does not reproduce Hazard_Prob_LGBM -> paste output to Claude"

# 6) training rows = exactly the OOF rows of v03
oof = pd.read_csv(V03 / "oof_v03.csv")
print("\noof_v03 columns:", oof.columns.tolist(), "| rows:", len(oof))
key = ["Grid_ID", "Event_ID"]
assert all(k in oof.columns for k in key)
blk_cols = [c for c in oof.columns if "block" in c.lower()]
tr = oof[key + blk_cols[:1]].drop_duplicates(key)
lab = pd.read_csv(LABELS, usecols=key + [TARGET])
tr = tr.merge(lab, on=key, how="left", validate="1:1")
tr = tr.merge(cells[["Grid_ID", "X_m", "Y_m"] + FEATURES], on="Grid_ID", how="left", validate="m:1")
assert tr[TARGET].notna().all() and set(tr[TARGET].unique()) <= {0, 1}
if blk_cols:
    tr = tr.rename(columns={blk_cols[0]: "Block_ID"})
    print("Block_ID taken from oof column:", blk_cols[0])
else:
    tr["Block_ID"] = (np.floor(tr["X_m"] / BLOCK_SIZE_M).astype(int).astype(str) + "_" +
                      np.floor(tr["Y_m"] / BLOCK_SIZE_M).astype(int).astype(str))
    print("Block_ID recomputed from centroids (15 km floor)")
n_rows, n_cells_tr, n_blk = len(tr), tr["Grid_ID"].nunique(), tr["Block_ID"].nunique()
print(f"training rows {n_rows} | cells {n_cells_tr} | blocks {n_blk} | positive rate {tr[TARGET].mean():.3f}")
assert n_rows == EXPECTED_TRAIN_ROWS, "training rows differ from Phase 3"
assert n_cells_tr == EXPECTED_TRAIN_CELLS
if n_blk != EXPECTED_BLOCKS:
    print(f"WARNING: {n_blk} blocks vs {EXPECTED_BLOCKS} in Phase 3 (block origin differs) -> tell Claude")
print("\nCELL 1 OK")

== final_v03 files ==
  ab_cell_level_target_v03.csv                                 0.0 MB
  ab_relelev_bootstrap_v03.csv                                 0.0 MB
  ab_relelev_metrics_v03.csv                                   0.0 MB
  bootstrap_ci_v03.csv                                         0.0 MB
  calibration_v03.csv                                          0.0 MB
  f2_bootstrap_v03.csv                                         0.0 MB
  f2_feature_metrics_v03.csv                                   0.0 MB
  f3_hand_twi_bootstrap_v03.csv                                0.0 MB
  f3_hand_twi_metrics_v03.csv                                  0.0 MB
  final_models_v03.joblib                                    242.9 MB
  hazard_predictions_v03.csv                                  24.3 MB
  loeo_v03.csv                                                 0.0 MB
  metrics_v03.csv                                              0.0 MB
  model_config_v03.json                                        0.0 M

In [3]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 2: TreeSHAP on final LGBM (raw log-odds, pre-calibration)
import shap

sv_sum = np.zeros((len(cells), len(FEATURES)))
base_sum = 0.0
for path, m in members:
    X = X_for(m, cells)
    expl = shap.TreeExplainer(m)
    sv = expl.shap_values(X)
    if isinstance(sv, list):
        sv = sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, -1]
    ev = float(np.atleast_1d(expl.expected_value)[-1])
    err = np.abs(sv.sum(axis=1) + ev - m.predict(X, raw_score=True)).max()
    print(f"{path}: additivity max error {err:.2e}")
    assert err < 1e-3
    sv_sum += sv
    base_sum += ev
SV = sv_sum / len(members)
BASE = base_sum / len(members)

SHAP_COLS = [f"SHAP_LogOdds_{f}" for f in FEATURES]
shap_df = pd.DataFrame(SV, columns=SHAP_COLS)
shap_df.insert(0, "Grid_ID", cells["Grid_ID"].values)
shap_df["SHAP_Base_LogOdds"] = BASE
F = np.array(FEATURES)
dom_idx = np.abs(SV).argmax(axis=1)
shap_df["Dominant_Driver"] = F[dom_idx]
shap_df["Dominant_Driver_Sign"] = np.where(SV[np.arange(len(SV)), dom_idx] > 0, "Increases", "Decreases")
pos = np.where(SV > 0, SV, 0.0)
shap_df["Dominant_Increasing_Driver"] = np.where(pos.max(axis=1) > 0, F[pos.argmax(axis=1)], "None")

aoa = cells["Is_Within_AOA"].to_numpy() == 1
imp = pd.DataFrame({
    "Feature": FEATURES,
    "Mean_Abs_SHAP_All": np.abs(SV).mean(0),
    "Mean_Abs_SHAP_AOA": np.abs(SV[aoa]).mean(0),
})
imp["Share_All_Pct"] = 100 * imp["Mean_Abs_SHAP_All"] / imp["Mean_Abs_SHAP_All"].sum()
imp["Share_AOA_Pct"] = 100 * imp["Mean_Abs_SHAP_AOA"] / imp["Mean_Abs_SHAP_AOA"].sum()
imp = imp.sort_values("Mean_Abs_SHAP_All", ascending=False)
print("== Global SHAP importance (log-odds) ==\n", imp.round(4).to_string(index=False))

tr_cells = tr.drop_duplicates("Grid_ID")
corr = tr_cells[FEATURES].corr(method="spearman")
print("\n== Spearman among features (training cells) ==\n", corr.round(3))
flags = [(a, b, corr.loc[a, b]) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:]
         if abs(corr.loc[a, b]) >= CORR_FLAG]
print("Correlation flags |rho| >= 0.7:", flags if flags else "none")

pop = cells["Population_Count"].to_numpy() if "Population_Count" in cells else None
drv = (shap_df.groupby("Dominant_Increasing_Driver").size() / len(shap_df) * 100).rename("Cells_Pct").to_frame()
if pop is not None:
    drv["Population_Pct"] = (pd.Series(pop).groupby(shap_df["Dominant_Increasing_Driver"]).sum()
                             / np.nansum(pop) * 100)
drv["Cells_Pct_Within_AOA"] = (shap_df[aoa].groupby("Dominant_Increasing_Driver").size() / aoa.sum() * 100)
print("\n== Dominant hazard-increasing driver ==\n", drv.round(1))

imp.to_csv(OUT / "shap_global_importance_v01.csv", index=False)
corr.to_csv(OUT / "feature_spearman_v01.csv")
drv.to_csv(OUT / "dominant_driver_share_v01.csv")
print("\nCELL 2 OK")

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\shap\explainers\_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


bundle['LGBM']: additivity max error 3.60e-14
== Global SHAP importance (log-odds) ==
             Feature  Mean_Abs_SHAP_All  Mean_Abs_SHAP_AOA  Share_All_Pct  Share_AOA_Pct
   Elevation_Mean_m             1.1774             0.5681        44.2290        33.0680
  Cropland_Fraction             1.0074             0.7614        37.8431        44.3149
     Slope_Mean_deg             0.3016             0.2431        11.3277        14.1504
Distance_to_River_m             0.1757             0.1455         6.6002         8.4668

== Spearman among features (training cells) ==
                      Elevation_Mean_m  Slope_Mean_deg  Distance_to_River_m  \
Elevation_Mean_m                1.000           0.296               -0.228   
Slope_Mean_deg                  0.296           1.000               -0.101   
Distance_to_River_m            -0.228          -0.101                1.000   
Cropland_Fraction              -0.218          -0.496                0.056   

                     Cropland_Fra

In [4]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 3: beeswarm, bar, dependence + sign-change values
import matplotlib
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)
idx_aoa = np.where(aoa)[0]
samp = rng.choice(idx_aoa, size=min(20000, len(idx_aoa)), replace=False)

shap.summary_plot(SV[samp], cells.loc[samp, FEATURES], show=False)
plt.title("SHAP (log-odds), cells within AOA, n=20,000 sample")
plt.savefig(FIG / "Fig_P4_SHAP_Beeswarm.png", dpi=300, bbox_inches="tight"); plt.close()

shap.summary_plot(SV[samp], cells.loc[samp, FEATURES], plot_type="bar", show=False)
plt.savefig(FIG / "Fig_P4_SHAP_Bar.png", dpi=300, bbox_inches="tight"); plt.close()

rows = []
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
for k, (f, ax) in enumerate(zip(FEATURES, axes.ravel())):
    x = cells.loc[aoa, f].to_numpy(float)
    s = SV[aoa, k]
    ok = np.isfinite(x)
    x, s = x[ok], s[ok]
    edges = np.unique(np.quantile(x, np.linspace(0, 1, N_BINS + 1)))
    b = np.clip(np.digitize(x, edges[1:-1]), 0, len(edges) - 2)
    g = (pd.DataFrame({"b": b, "x": x, "s": s}).groupby("b")
         .agg(x_med=("x", "median"), s_mean=("s", "mean"), n=("s", "size")))
    xs, ss = g["x_med"].to_numpy(), g["s_mean"].to_numpy()
    cross = [xs[i] - ss[i] * (xs[i + 1] - xs[i]) / (ss[i + 1] - ss[i])
             for i in range(len(ss) - 1) if ss[i] != 0 and np.sign(ss[i]) != np.sign(ss[i + 1])]
    rows.append({"Feature": f, "N_Cells_AOA": len(x),
                 "Sign_Change_Values": "; ".join(f"{c:.4g}" for c in cross) if cross else "none",
                 "Bin_Mean_SHAP_Min": ss.min(), "Bin_Mean_SHAP_Max": ss.max()})
    sub = rng.choice(len(x), size=min(15000, len(x)), replace=False)
    ax.scatter(x[sub], s[sub], s=2, alpha=0.15, color="grey", rasterized=True)
    ax.plot(xs, ss, color="crimson", lw=2, label="binned mean")
    ax.axhline(0, color="k", lw=0.8)
    for c in cross:
        ax.axvline(c, ls="--", color="navy", lw=1)
    ax.set_xlim(np.percentile(x, 0.5), np.percentile(x, 99.5))
    ax.set_xlabel(f); ax.set_ylabel("SHAP (log-odds)")
fig.suptitle("SHAP dependence within AOA (dashed = mean SHAP changes sign; model behaviour, not causation)")
fig.tight_layout(); fig.savefig(FIG / "Fig_P4_SHAP_Dependence.png", dpi=300); plt.close(fig)

thr = pd.DataFrame(rows)
thr.to_csv(OUT / "shap_sign_change_values_v01.csv", index=False)
print(thr.round(4).to_string(index=False))
print("\nCELL 3 OK")

C:\Users\User\AppData\Local\Temp\ipykernel_18988\3091534948.py:9: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(SV[samp], cells.loc[samp, FEATURES], show=False)
C:\Users\User\AppData\Local\Temp\ipykernel_18988\3091534948.py:13: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(SV[samp], cells.loc[samp, FEATURES], plot_type="bar", show=False)


            Feature  N_Cells_AOA                    Sign_Change_Values  Bin_Mean_SHAP_Min  Bin_Mean_SHAP_Max
   Elevation_Mean_m        86151                          1.629; 2.217            -3.1869             0.4397
     Slope_Mean_deg        86151                               0.08172            -0.6737             0.4255
Distance_to_River_m        86151 141.2; 490.2; 928.9; 1005; 1360; 3584            -0.7168             0.1476
  Cropland_Fraction        86151                                0.8775            -1.5456             0.8243

CELL 3 OK


In [5]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 4: block-bootstrap ensemble -> rank uncertainty + driver stability
import time
from scipy.stats import rankdata

template = members[0][1]
params = template.get_params()
params.update({"verbose": -1, "n_jobs": -1})
print("template params:", {k: v for k, v in params.items() if v is not None})
print("template trees:", template.booster_.num_trees())

blk = tr["Block_ID"].astype(str).to_numpy()
ublk = np.unique(blk)
blk_rows = {b: np.where(blk == b)[0] for b in ublk}
Xtr = tr[FEATURES]
ytr = tr[TARGET].astype(int).to_numpy()
Xall = cells[FEATURES]
n, N = len(cells), N_ENSEMBLE

P = np.empty((n, N), np.float32)
R = np.empty((n, N), np.float32)
top10 = np.zeros(n, np.int32)
same_drv = np.zeros(n, np.int32)
rng = np.random.default_rng(SEED)
t0 = time.time()
for b in range(N):
    pick = rng.choice(ublk, size=len(ublk), replace=True)
    rows_b = np.concatenate([blk_rows[k] for k in pick])
    assert ytr[rows_b].sum() > 0
    p = dict(params); p["random_state"] = SEED + b
    mb = lgb.LGBMClassifier(**p).fit(Xtr.iloc[rows_b], ytr[rows_b])
    prob = mb.predict_proba(Xall)[:, 1]
    contrib = mb.predict(Xall, pred_contrib=True)[:, :len(FEATURES)]
    r = rankdata(prob) / n * 100
    P[:, b], R[:, b] = prob, r
    top10 += (r > 90)
    same_drv += (np.abs(contrib).argmax(axis=1) == dom_idx)
    if (b + 1) % 10 == 0:
        print(f"  member {b + 1}/{N} done ({time.time() - t0:.0f} s)")

cells["Hazard_Rank_Pct_LGBM"] = rankdata(cells["Hazard_Prob_LGBM"]) / n * 100
ens = pd.DataFrame({
    "Grid_ID": cells["Grid_ID"].values,
    "Hazard_Rank_Pct_LGBM": cells["Hazard_Rank_Pct_LGBM"].values,
    "Hazard_Ens_Mean_Prob_Raw": P.mean(1),
    "Hazard_Ens_SD_Prob_Raw": P.std(1, ddof=1),
    "Hazard_Ens_Rank_Mean_Pct": R.mean(1),
    "Hazard_Ens_Rank_SD_Pct": R.std(1, ddof=1),
    "Hazard_Ens_Rank_P05_Pct": np.percentile(R, 5, axis=1),
    "Hazard_Ens_Rank_P95_Pct": np.percentile(R, 95, axis=1),
    "Top10_Frequency_Pct": top10 / N * 100,
    "Driver_Stability_Pct": same_drv / N * 100,
})

rho_ens = spearmanr(ens["Hazard_Rank_Pct_LGBM"], ens["Hazard_Ens_Rank_Mean_Pct"]).correlation
obs = cells["Is_Observable"].to_numpy() == 1
print(f"\nSpearman(final rank, ensemble mean rank) = {rho_ens:.3f}")
print(f"Rank SD (pct points) median: all {ens.Hazard_Ens_Rank_SD_Pct.median():.2f} | "
      f"within AOA {ens.Hazard_Ens_Rank_SD_Pct[aoa].median():.2f} | outside {ens.Hazard_Ens_Rank_SD_Pct[~aoa].median():.2f} | "
      f"observable {ens.Hazard_Ens_Rank_SD_Pct[obs].median():.2f} | unobservable {ens.Hazard_Ens_Rank_SD_Pct[~obs].median():.2f}")
print(f"Driver stability: mean {ens.Driver_Stability_Pct.mean():.1f}% | cells >= 80%: "
      f"{(ens.Driver_Stability_Pct >= 80).mean() * 100:.1f}% | within AOA mean {ens.Driver_Stability_Pct[aoa].mean():.1f}%")
final_top10 = ens["Hazard_Rank_Pct_LGBM"] > 90
print(f"Final top-10% cells kept in top 10% by >= 80% of members: "
      f"{(ens.Top10_Frequency_Pct[final_top10] >= 80).mean() * 100:.1f}%")

phase4 = shap_df.merge(ens, on="Grid_ID", validate="1:1")
assert len(phase4) == EXPECTED_MODEL_CELLS and phase4["Grid_ID"].is_unique
phase4.to_csv(OUT / "phase4_shap_ensemble_v01.csv", index=False)
np.savez_compressed(OUT / "ensemble_members_prob_v01.npz", Grid_ID=cells["Grid_ID"].to_numpy(), P=P)
(OUT / "ensemble_config_v01.json").write_text(json.dumps(
    {"n_members": N, "seed": SEED, "bootstrap": "15 km spatial blocks with replacement",
     "n_blocks": int(len(ublk)), "features": FEATURES, "params": {k: str(v) for k, v in params.items()},
     "note": "members uncalibrated; uncertainty reported on rank percentiles"}, indent=1))
print("\nCELL 4 OK")

template params: {'boosting_type': 'gbdt', 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.03, 'max_depth': -1, 'min_child_samples': 50, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': -1, 'num_leaves': 31, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.8, 'subsample_for_bin': 200000, 'subsample_freq': 1, 'verbose': -1}
template trees: 500
  member 10/50 done (1769 s)
  member 20/50 done (3533 s)
  member 30/50 done (5135 s)
  member 40/50 done (5931 s)
  member 50/50 done (6809 s)

Spearman(final rank, ensemble mean rank) = 0.992
Rank SD (pct points) median: all 5.35 | within AOA 5.11 | outside 5.53 | observable 5.05 | unobservable 5.41
Driver stability: mean 83.8% | cells >= 80%: 70.4% | within AOA mean 83.4%
Final top-10% cells kept in top 10% by >= 80% of members: 68.7%

CELL 4 OK


In [6]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 5: maps (polygon plots, EPSG:32646)
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

v02 = pd.read_csv(V02_PRED, usecols=["Grid_ID", "Hazard_Prob_LGBM"]).rename(
    columns={"Hazard_Prob_LGBM": "Hazard_Prob_LGBM_v02"})
assert v02["Grid_ID"].is_unique
mp = phase4.merge(cells[["Grid_ID", "Hazard_Prob_LGBM", "Is_Within_AOA"]], on="Grid_ID") \
           .merge(v02, on="Grid_ID", how="left", validate="1:1")
assert mp["Hazard_Prob_LGBM_v02"].notna().all()
mp["Hazard_Rank_Pct_v02"] = rankdata(mp["Hazard_Prob_LGBM_v02"]) / len(mp) * 100
mp["Rank_Change_v03_minus_v02"] = mp["Hazard_Rank_Pct_LGBM"] - mp["Hazard_Rank_Pct_v02"]

g_model = grid[["Grid_ID", "geometry"]].merge(mp, on="Grid_ID", how="inner")
g_water = grid[["Grid_ID", "geometry"]].merge(master.loc[master["Is_Water"] == 1, ["Grid_ID"]], on="Grid_ID")
assert len(g_model) == EXPECTED_MODEL_CELLS

def panel(ax, col, title, cmap="viridis", vmin=None, vmax=None, cats=None):
    g_water.plot(ax=ax, color="#d9e6f2", linewidth=0, rasterized=True)
    if cats:
        cm = ListedColormap([c for _, c in cats])
        codes = g_model[col].map({k: i for i, (k, _) in enumerate(cats)})
        g_model.assign(_c=codes).plot(ax=ax, column="_c", cmap=cm, vmin=-0.5, vmax=len(cats) - 0.5,
                                      linewidth=0, rasterized=True)
        ax.legend(handles=[Patch(color=c, label=k) for k, c in cats], loc="lower left", fontsize=7)
    else:
        g_model.plot(ax=ax, column=col, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0,
                     legend=True, legend_kwds={"shrink": 0.6}, rasterized=True)
    ax.set_title(title, fontsize=10); ax.set_axis_off()

fig, ax = plt.subplots(1, 3, figsize=(18, 7))
panel(ax[0], "Hazard_Prob_LGBM", "Flood hazard probability (calibrated LGBM)", "YlOrRd")
panel(ax[1], "Hazard_Ens_Rank_SD_Pct", "Ensemble uncertainty (SD of rank, pct points)", "magma_r")
g_model["AOA_Label"] = np.where(g_model["Is_Within_AOA"] == 1, "Within AOA", "Outside AOA")
panel(ax[2], "AOA_Label", "Area of applicability", cats=[("Within AOA", "#1b9e77"), ("Outside AOA", "#bdbdbd")])
fig.tight_layout(); fig.savefig(FIG / "Fig_P4_Map_Hazard_Uncertainty_AOA.png", dpi=300); plt.close(fig)

drv_colors = [("Elevation_Mean_m", "#e41a1c"), ("Slope_Mean_deg", "#984ea3"),
              ("Distance_to_River_m", "#377eb8"), ("Cropland_Fraction", "#4daf4a"), ("None", "#f0f0f0")]
fig, ax = plt.subplots(1, 2, figsize=(13, 7))
panel(ax[0], "Dominant_Increasing_Driver", "Dominant hazard-increasing driver (SHAP)", cats=drv_colors)
panel(ax[1], "Driver_Stability_Pct", "Driver stability across 50 members (%)", "cividis", 0, 100)
fig.tight_layout(); fig.savefig(FIG / "Fig_P4_Map_Driver_Stability.png", dpi=300); plt.close(fig)

fig, ax = plt.subplots(1, 3, figsize=(18, 7))
panel(ax[0], "Hazard_Rank_Pct_v02", "v02 naive labels: hazard percentile", "YlOrRd", 0, 100)
panel(ax[1], "Hazard_Rank_Pct_LGBM", "v03 observability-aware: hazard percentile", "YlOrRd", 0, 100)
panel(ax[2], "Rank_Change_v03_minus_v02", "Percentile change (v03 - v02)", "RdBu_r", -60, 60)
fig.tight_layout(); fig.savefig(FIG / "Fig_P4_Map_v02_vs_v03.png", dpi=300); plt.close(fig)
print("Figures saved:", sorted(p.name for p in FIG.glob("*.png")))
print("\nCELL 5 OK")

Figures saved: ['Fig_P4_Map_Driver_Stability.png', 'Fig_P4_Map_Hazard_Uncertainty_AOA.png', 'Fig_P4_Map_v02_vs_v03.png', 'Fig_P4_SHAP_Bar.png', 'Fig_P4_SHAP_Beeswarm.png', 'Fig_P4_SHAP_Dependence.png']

CELL 5 OK


In [7]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 6: paste-back summary
print("=== PHASE 4 SUMMARY ===")
print("LGBM members used:", len(members), "| reproduction rho:", round(rho_rep, 4))
print("Global importance:\n", imp[["Feature", "Share_All_Pct", "Share_AOA_Pct"]].round(1).to_string(index=False))
print("Correlation flags:", flags if flags else "none")
print("Dominant increasing driver:\n", drv.round(1).to_string())
print("Sign-change values:\n", thr[["Feature", "Sign_Change_Values"]].to_string(index=False))
print(f"Ensemble: rho(final, mean rank) {rho_ens:.3f}; median rank SD {ens.Hazard_Ens_Rank_SD_Pct.median():.2f}; "
      f"in/out AOA {ens.Hazard_Ens_Rank_SD_Pct[aoa].median():.2f}/{ens.Hazard_Ens_Rank_SD_Pct[~aoa].median():.2f}")
print(f"Driver stability mean {ens.Driver_Stability_Pct.mean():.1f}%; top-10% robust (>=80% members) "
      f"{(ens.Top10_Frequency_Pct[final_top10] >= 80).mean() * 100:.1f}%")
print(f"Spearman rank change v02->v03 vs Tree? (Phase 6) | mean |rank change| {mp.Rank_Change_v03_minus_v02.abs().mean():.1f} pts")

=== PHASE 4 SUMMARY ===
LGBM members used: 1 | reproduction rho: 1.0
Global importance:
             Feature  Share_All_Pct  Share_AOA_Pct
   Elevation_Mean_m           44.2           33.1
  Cropland_Fraction           37.8           44.3
     Slope_Mean_deg           11.3           14.2
Distance_to_River_m            6.6            8.5
Correlation flags: none
Dominant increasing driver:
                             Cells_Pct  Population_Pct  Cells_Pct_Within_AOA
Dominant_Increasing_Driver                                                 
Cropland_Fraction                12.6             3.7                  22.6
Distance_to_River_m              24.4            29.1                  16.3
Elevation_Mean_m                 24.9            26.3                  35.4
None                             23.0            28.5                  15.5
Slope_Mean_deg                   15.1            12.4                  10.2
Sign-change values:
             Feature                    Sign_Change_Valu

In [8]:
import pandas as pd
cols = list(pd.read_csv(MASTER, nrows=0).columns)
print(len(cols), "columns in master_v06:")
for c in cols:
    print(" ", c)

65 columns in master_v06:
  Grid_ID
  District
  Upazila
  Latitude
  Longitude
  Centroid_X_m
  Centroid_Y_m
  Grid_Area_km2
  Frac_Sum
  Water_Fraction
  Tree_Fraction
  Cropland_Fraction
  Built_Fraction
  Mangrove_Fraction
  Is_Water
  Land_Cover_Code
  Land_Cover_Class
  Elevation_Mean_m
  Elevation_Valid_Fraction
  Annual_Rainfall_mm
  Rain_Filled_Flag
  LST_Day_Mean_C
  LST_Filled_Flag
  NDVI_Mean
  NDVI_Valid_Fraction
  Population_Count
  Population_Density_per_km2
  Building_Count
  Building_Density_per_km2
  Building_Footprint_Fraction
  Slope_Mean_deg
  Slope_Valid_Fraction
  Distance_to_River_m
  Distance_to_Coast_m
  Distance_to_Waterway_m
  Road_Density_km_per_km2
  Distance_to_Road_m
  Distance_to_Hospital_m
  Distance_to_Embankment_m
  Baseline_Air_Temp_NEX_C
  Rainfall_CF_SSP245
  Rainfall_CF_SSP245_Std
  Future_Rainfall_SSP245_mm
  Temp_Delta_SSP245_C
  Temp_Delta_SSP245_C_Std
  Future_Temperature_SSP245_C
  Rainfall_CF_SSP585
  Rainfall_CF_SSP585_Std
  Future_Rainfal

In [9]:
# Notebook: P4_SHAP_Ensemble_v01.ipynb | Cell 8: diagnostic - where v03 lowered hazard, and do observed labels support it?
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)

ROOT = Path(r"D:\Siam\Thesis_GeoAI")
MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
V03 = ROOT / "05_models" / "final_v03"
V02_PRED = ROOT / "05_models" / "final_v02" / "hazard_predictions_v02.csv"
P4 = ROOT / "05_models" / "phase4_v01" / "phase4_shap_ensemble_v01.csv"
OUT = ROOT / "05_models" / "phase4_v01"
N_MODEL = 160874

mcols = ["Grid_ID", "District", "Upazila_Name", "Is_Water", "Population_Count", "Elevation_Mean_m",
         "Cropland_Fraction", "Tree_Fraction", "Mangrove_Fraction_GMW", "Water_Fraction",
         "Built_Fraction", "Land_Cover_Class"]
m = pd.read_csv(MASTER, usecols=mcols)
m = m[m["Is_Water"] == 0]
hz = pd.read_csv(V03 / "hazard_predictions_v03.csv", usecols=["Grid_ID", "Is_Within_AOA", "Is_Observable"])
p4 = pd.read_csv(P4, usecols=["Grid_ID", "Hazard_Rank_Pct_LGBM", "Dominant_Increasing_Driver"])
v02 = pd.read_csv(V02_PRED, usecols=["Grid_ID", "Hazard_Prob_LGBM"])
assert len(v02) == N_MODEL and v02["Grid_ID"].is_unique
v02["Hazard_Rank_Pct_v02"] = v02["Hazard_Prob_LGBM"].rank(pct=True) * 100

df = (m.merge(hz, on="Grid_ID", validate="1:1")
       .merge(p4, on="Grid_ID", validate="1:1")
       .merge(v02[["Grid_ID", "Hazard_Rank_Pct_v02"]], on="Grid_ID", validate="1:1"))
assert len(df) == N_MODEL, len(df)
assert df["Upazila_Name"].notna().all(), "Upazila_Name has NaN -> tell Claude"
df["Rank_Change"] = df["Hazard_Rank_Pct_LGBM"] - df["Hazard_Rank_Pct_v02"]

# Table A: per upazila - hazard change, landscape, observed vs predicted flood rate (OOF, observable training rows)
g = df.groupby(["District", "Upazila_Name"])
A = pd.DataFrame({
    "Cells": g.size(),
    "Pop_k": (g["Population_Count"].sum() / 1000).round(1),
    "Rank_v02": g["Hazard_Rank_Pct_v02"].mean().round(1),
    "Rank_v03": g["Hazard_Rank_Pct_LGBM"].mean().round(1),
    "Change": g["Rank_Change"].mean().round(1),
    "Out_AOA_Pct": g["Is_Within_AOA"].apply(lambda s: (s == 0).mean() * 100).round(1),
    "Unobs_Pct": g["Is_Observable"].apply(lambda s: (s == 0).mean() * 100).round(1),
    "Crop": g["Cropland_Fraction"].mean().round(2),
    "Tree": g["Tree_Fraction"].mean().round(2),
    "Mang": g["Mangrove_Fraction_GMW"].mean().round(2),
    "WaterF": g["Water_Fraction"].mean().round(2),
    "Built": g["Built_Fraction"].mean().round(2),
    "Top_LC": g["Land_Cover_Class"].agg(lambda s: s.value_counts().index[0]),
}).reset_index()

oof = pd.read_csv(V03 / "oof_v03.csv", usecols=["Grid_ID", "Event_ID", "y", "LGBM"])
oof = oof.merge(df[["Grid_ID", "Upazila_Name", "Elevation_Mean_m", "Water_Fraction",
                    "Cropland_Fraction", "Tree_Fraction"]], on="Grid_ID", how="left", validate="m:1")
assert oof["Upazila_Name"].notna().all()
go = oof.groupby("Upazila_Name")
B = pd.DataFrame({"Train_Rows": go.size(), "Obs_Rate": go["y"].mean().round(3),
                  "OOF_Pred": go["LGBM"].mean().round(3)}).reset_index()
B["Obs_to_Pred"] = (B["Obs_Rate"] / B["OOF_Pred"]).round(2)
A = A.merge(B, on="Upazila_Name", how="left").sort_values("Change")
print("== Table A: per upazila (sorted by v03 - v02 change) ==")
print(A.to_string(index=False))
print(f"\nStudy-wide observed rate {oof['y'].mean():.3f} | mean OOF pred {oof['LGBM'].mean():.3f}")

# Table B: elevation bins - is the negative SHAP below 1.63 m supported by observed labels?
bins = [-np.inf, 1.0, 1.629, 2.217, 3.0, np.inf]
labs = ["<1.0", "1.0-1.63", "1.63-2.22", "2.22-3.0", ">3.0"]
oof["Elev_Bin"] = pd.cut(oof["Elevation_Mean_m"], bins=bins, labels=labs)
C = oof.groupby("Elev_Bin", observed=True).agg(
    Rows=("y", "size"), Obs_Rate=("y", "mean"), OOF_Pred=("LGBM", "mean"),
    WaterF=("Water_Fraction", "mean"), Crop=("Cropland_Fraction", "mean"), Tree=("Tree_Fraction", "mean")).round(3)
print("\n== Table B: training rows by elevation bin ==")
print(C.to_string())

# Table C: rank association with land-cover fractions (all model cells)
print("\n== Spearman of hazard rank with land cover (v02 -> v03) ==")
for c in ["Cropland_Fraction", "Tree_Fraction", "Built_Fraction", "Water_Fraction", "Mangrove_Fraction_GMW"]:
    r2 = spearmanr(df["Hazard_Rank_Pct_v02"], df[c]).correlation
    r3 = spearmanr(df["Hazard_Rank_Pct_LGBM"], df[c]).correlation
    print(f"  {c:24s} v02 {r2:+.3f} -> v03 {r3:+.3f}")

A.to_csv(OUT / "diag_upazila_v01.csv", index=False)
C.to_csv(OUT / "diag_elevation_bins_v01.csv")
print("\nCELL 8 OK")

== Table A: per upazila (sorted by v03 - v02 change) ==
  District            Upazila_Name  Cells  Pop_k  Rank_v02  Rank_v03  Change  Out_AOA_Pct  Unobs_Pct  Crop  Tree  Mang  WaterF  Built     Top_LC  Train_Rows  Obs_Rate  OOF_Pred  Obs_to_Pred
  Bagerhat                  Mongla  17959  180.1      41.8      13.3   -28.4         80.7       96.0  0.03  0.04  0.89    0.04   0.00  Mangroves         861     0.009     0.018         0.50
  Bagerhat             Sharankhola  10170  124.1      42.7      23.8   -18.9         78.1       90.0  0.12  0.09  0.75    0.02   0.00  Mangroves        3000     0.491     0.235         2.09
  Bagerhat                  Rampal   3007  204.3      55.6      42.9   -12.7         42.8       48.5  0.42  0.36  0.00    0.12   0.01   Cropland        2226     0.082     0.091         0.90
  Bagerhat                Mollahat   2946  147.2      56.7      48.1    -8.7         47.5       39.3  0.61  0.30  0.00    0.03   0.00   Cropland        4170     0.007     0.098        